# Notebook 3 — Verb Extraction & Caption Masking

Produces ground-truth verb labels from `annotations/segments_available.csv`
using a two-stage spaCy dependency-parsing pipeline:

1. **Stage 1 — spaCy ROOT parsing**: extract the syntactic head verb of each caption
2. **Stage 2 — Auxiliary filtering + lemmatization**: remove light/auxiliary verbs, normalise surface forms
3. **Masking**: replace the first occurrence of the extracted verb in the caption with `[MASK]`
4. **QA report**: extraction rate, top verbs, and a 100-row spot-check sample to validate manually

**Why spaCy ROOT parsing?**  
YouCook2 captions are short, single-clause, imperative sentences ("Add the onions", "Stir until combined").
For this structure the syntactic ROOT is almost always the main action verb — exactly what we want as our label.
SRL adds complexity and latency with no measurable gain on imperative cooking captions (see Xu et al., 2020).
The method is deterministic, reproducible, and fully citable.

**Outputs:**
- `annotations/segments_verbs.csv` — full table with `extracted_verb` and `masked_caption` columns
- `annotations/spotcheck_sample.csv` — 100-row manual validation sample
- `annotations/verb_extraction_report.txt` — stats for your paper's methods section


## 0. Install & import dependencies

In [2]:
import subprocess, sys

# Install spaCy and the English model if not already present
# subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'spacy'], check=True)
# subprocess.run([sys.executable, '-m', 'spacy', 'download', 'en_core_web_sm', '--quiet'], check=True)

import json, os, re
import pandas as pd
import spacy
from collections import Counter
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

with open('/home/ubuntu/youcook2-videos/paths.json') as f:
    PATHS = json.load(f)

nlp = spacy.load('en_core_web_sm', disable=['ner'])
print('spaCy version:', spacy.__version__)
print('Model loaded  : en_core_web_sm')

spaCy version: 3.8.14
Model loaded  : en_core_web_sm


## 1. Load segments

In [3]:
csv_path = os.path.join(PATHS['annotations'], 'segments_available.csv')
df = pd.read_csv(csv_path)

print(f'Segments loaded : {len(df):,}')
print(f'Unique videos   : {df["video_id"].nunique():,}')
print(f'Columns         : {list(df.columns)}')
df.head(3)


Segments loaded : 13,716
Unique videos   : 1,774
Columns         : ['video_id', 'subset', 'recipe_type', 'segment_id', 'ann_id', 'start_sec', 'end_sec', 'duration', 'caption']


,video_id,subset,recipe_type,segment_id,ann_id,start_sec,end_sec,duration,caption
0,GLd3aX16zBg,training,113,GLd3aX16zBg_000,0,90.0,102.0,12.0,spread margarine on two slices of white bread
1,GLd3aX16zBg,training,113,GLd3aX16zBg_001,1,114.0,127.0,13.0,place a slice of cheese on the bread
2,GLd3aX16zBg,training,113,GLd3aX16zBg_002,2,132.0,138.0,6.0,place the bread slices on top of each other an...


## 3. Run extraction over all segments

Uses spaCy's `nlp.pipe()` for batched, efficient processing.

In [4]:
def extract_verb(caption: str) -> tuple[str | None, str]:
    """
    Extract the main action verb from a YouCook2 caption using
    spaCy's dependency parse exclusively — no manual lists.

    spaCy's dep_ tags used:
      ROOT   — syntactic head of the sentence
      aux    — auxiliary verb (e.g. 'will', 'have', 'do')
      auxpass — passive auxiliary (e.g. 'been', 'was')

    Returns
    -------
    (lemma, method)  where method is one of:
        'root'     — syntactic ROOT token
        'fallback' — first VERB not tagged as auxiliary
        'none'     — no verb found
    """
    doc = nlp(caption.strip())

    # Stage 1: ROOT token that spaCy itself classifies as VERB
    # and whose dependency is not auxiliary in nature
    for token in doc:
        if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
            return token.lemma_.lower(), 'root'

    # Stage 2: Fallback — first token spaCy tags as VERB
    # that is NOT tagged as an auxiliary by the parser
    for token in doc:
        if token.pos_ == 'VERB' and token.dep_ not in ('aux', 'auxpass'):
            return token.lemma_.lower(), 'fallback'

    return None, 'none'


def apply_mask(caption: str, verb: str | None) -> str:
    """
    Replace the first surface-form occurrence of `verb`
    in `caption` with [MASK], using spaCy's token offsets.
    """
    if verb is None:
        return caption

    doc = nlp(caption.strip())
    for token in doc:
        if token.lemma_.lower() == verb and token.pos_ == 'VERB':
            return caption[:token.idx] + '[MASK]' + caption[token.idx + len(token.text):]

    return caption  # no regex fallback needed — spaCy found it or it's not there


# Sanity check
examples = [
    'Add the chopped onions to the pan.',
    'Stir the mixture until it thickens.',
    'Once the oil is hot, carefully place the fish in the pan.',
]
for ex in examples:
    verb, method = extract_verb(ex)
    masked = apply_mask(ex, verb)
    print(f'Caption : {ex}')
    print(f'Verb    : {verb!r}  [{method}]')
    print(f'Masked  : {masked}')
    print()

Caption : Add the chopped onions to the pan.
Verb    : 'add'  [root]
Masked  : [MASK] the chopped onions to the pan.

Caption : Stir the mixture until it thickens.
Verb    : 'stir'  [root]
Masked  : [MASK] the mixture until it thickens.

Caption : Once the oil is hot, carefully place the fish in the pan.
Verb    : 'place'  [root]
Masked  : Once the oil is hot, carefully [MASK] the fish in the pan.



In [7]:
from tqdm.notebook import tqdm

captions = df['caption'].tolist()
verbs, methods, masked_captions = [], [], []

BATCH = 256
docs = list(tqdm(
    nlp.pipe(captions, batch_size=BATCH),
    total=len(captions),
    desc='Parsing captions'
))

for doc, caption in zip(docs, captions):
    # Stage 1: ROOT token that spaCy classifies as VERB
    verb, method = None, 'none'
    for token in doc:
        if token.dep_ == 'ROOT' and token.pos_ == 'VERB':
            verb, method = token.lemma_.lower(), 'root'
            break

    # Stage 2: Fallback — first VERB not tagged as auxiliary by spaCy
    if verb is None:
        for token in doc:
            if token.pos_ == 'VERB' and token.dep_ not in ('aux', 'auxpass'):
                verb, method = token.lemma_.lower(), 'fallback'
                break

    verbs.append(verb)
    methods.append(method)

    # Masking — use spaCy token offsets directly, no regex needed
    masked = caption
    if verb:
        for token in doc:
            if token.lemma_.lower() == verb and token.pos_ == 'VERB':
                masked = caption[:token.idx] + '[MASK]' + caption[token.idx + len(token.text):]
                break
    masked_captions.append(masked)

df['extracted_verb']    = verbs
df['extraction_method'] = methods
df['masked_caption']    = masked_captions

n_extracted = df['extracted_verb'].notna().sum()
n_root      = (df['extraction_method'] == 'root').sum()
n_fallback  = (df['extraction_method'] == 'fallback').sum()
n_none      = (df['extraction_method'] == 'none').sum()

print(f'Total segments      : {len(df):,}')
print(f'Verb extracted      : {n_extracted:,}  ({n_extracted/len(df)*100:.1f}%)')
print(f'  via ROOT          : {n_root:,}')
print(f'  via fallback      : {n_fallback:,}')
print(f'No verb found       : {n_none:,}  ({n_none/len(df)*100:.1f}%)')

Parsing captions:   0%|          | 0/13716 [00:00<?, ?it/s]

Total segments      : 13,716
Verb extracted      : 13,029  (95.0%)
  via ROOT          : 12,572
  via fallback      : 457
No verb found       : 687  (5.0%)


In [ ]:
#choosing to ignore the no verb found

## 4. Inspection — sample masked captions

In [6]:
pd.set_option('display.max_colwidth', 120)

sample = df[df['extracted_verb'].notna()].sample(12, random_state=42)
sample[['video_id', 'caption', 'extracted_verb', 'extraction_method', 'masked_caption']]


,video_id,caption,extracted_verb,extraction_method,masked_caption
10367,a5FoLWnEiAI,mix dal makhni masala red chili powder and turmeric powder in a small bowl of water,mix,root,[MASK] dal makhni masala red chili powder and turmeric powder in a small bowl of water
398,7-WEdqJBXoQ,coat the chicken with the bread crumb powder and lay them on a baking pan,coat,root,[MASK] the chicken with the bread crumb powder and lay them on a baking pan
6328,7E8Lj_Ktfok,mix the apple and lime juice,mix,root,[MASK] the apple and lime juice
3875,3AAdKl1UYZs,absorb some of the fat with a paper towel,absorb,root,[MASK] some of the fat with a paper towel
4767,9GIPE0aeVNI,chop some lettuce and it add to the bowl,chop,root,[MASK] some lettuce and it add to the bowl
6750,fq37BTAjPHk,preheat the oven to 375 degrees and heat peanut oil in a pan,preheat,root,[MASK] the oven to 375 degrees and heat peanut oil in a pan
12583,LaouC0RQoDg,cut green onion and place on plate,cut,root,[MASK] green onion and place on plate
11190,r1HcLnSTJHY,pour gelas air soda dingin to remaining flour mixture in the bowl and whisk to form a thick batter,pour,root,[MASK] gelas air soda dingin to remaining flour mixture in the bowl and whisk to form a thick batter
879,P8SY69LYkx0,to make small burgers take small quantity of the mixture roll it and press so as to get small burger,make,fallback,to [MASK] small burgers take small quantity of the mixture roll it and press so as to get small burger
2336,TO_W2RYL2mA,add one chicken stock cube,add,root,[MASK] one chicken stock cube


## 6. Per-video breakdown

Organised view: video → all captions → masked caption + label

In [8]:
# Show one full video's worth of segments as a formatted table
sample_vid = df['video_id'].value_counts().index[0]  # most-segment video

vid_df = df[df['video_id'] == sample_vid][
    ['segment_id', 'start_sec', 'end_sec', 'caption', 'masked_caption', 'extracted_verb']
].reset_index(drop=True)

print(f'Video: {sample_vid}  |  {len(vid_df)} segments')
print('=' * 100)
vid_df


Video: qmoqJRP5N6E  |  16 segments


,segment_id,start_sec,end_sec,caption,masked_caption,extracted_verb
0,qmoqJRP5N6E_000,25.0,28.0,cook ramen noodles in water for 3 minutes and drain the noodles,[MASK] ramen noodles in water for 3 minutes and drain the noodles,cook
1,qmoqJRP5N6E_001,34.0,36.0,crack 1 egg in a bowl and whisk it thoroughly,[MASK] 1 egg in a bowl and whisk it thoroughly,crack
2,qmoqJRP5N6E_002,37.0,40.0,add the cooked noodles to the whisked egg and give a quick toss,[MASK] the cooked noodles to the whisked egg and give a quick toss,add
3,qmoqJRP5N6E_003,50.0,61.0,pour noodles into pan wrap cover press it using another pan remove it and keep in fridge for 15 minutes,[MASK] noodles into pan wrap cover press it using another pan remove it and keep in fridge for 15 minutes,pour
4,qmoqJRP5N6E_004,67.0,70.0,repeat the procedure again,[MASK] the procedure again,repeat
5,qmoqJRP5N6E_005,73.0,80.0,mix beef meat noodles tastemaker chopped spring onion soy sauce and sesame oil in a bowl,[MASK] beef meat noodles tastemaker chopped spring onion soy sauce and sesame oil in a bowl,mix
6,qmoqJRP5N6E_006,81.0,105.0,pack the seasoned meat into a burger shaped pan given fine burger shape patting by fingers,[MASK] the seasoned meat into a burger shaped pan given fine burger shape patting by fingers,pack
7,qmoqJRP5N6E_007,107.0,111.0,take the meat off the pan to get burger shape,[MASK] the meat off the pan to get burger shape,take
8,qmoqJRP5N6E_008,114.0,117.0,fry them in a pan on either side,[MASK] them in a pan on either side,fry
9,qmoqJRP5N6E_009,160.0,163.0,fry the noodles burger in another pan on either sides,[MASK] the noodles burger in another pan on either sides,fry


## 7. Grouped chart — video → captions → verbs

Compact summary table grouped by `video_id` for mentor presentation.

In [9]:
from IPython.display import display, HTML

# Pick a representative subset of videos (first 8 by appearance)
preview_vids = df['video_id'].unique()[:8]
preview_df   = df[df['video_id'].isin(preview_vids)].copy()

def build_html_table(df_subset):
    rows = []
    rows.append("""
    <style>
      .vt { font-family: 'DM Mono', monospace; border-collapse: collapse; width: 100%; font-size: 13px; }
      .vt th { background: #1a1f2e; color: #9098b8; font-weight: 500;
               padding: 8px 14px; text-align: left; border-bottom: 1px solid #252a38;
               font-size: 11px; text-transform: uppercase; letter-spacing: .6px; }
      .vt td { padding: 9px 14px; border-bottom: 1px solid #1c2030; vertical-align: top;
               color: #c8ccde; background: #13161e; }
      .vt tr:hover td { background: #191d28; }
      .vid-id { color: #5b8dff; font-weight: 600; white-space: nowrap; }
      .mask-tok { background: #2a1f00; color: #f0c040; border: 1px solid rgba(240,192,64,.3);
                  border-radius: 3px; padding: 1px 5px; font-size: 12px; }
      .verb-badge { background: rgba(240,192,64,.12); color: #f0c040;
                    border: 1px solid rgba(240,192,64,.25); border-radius: 5px;
                    padding: 2px 8px; font-size: 11px; white-space: nowrap; }
      .muted { color: #555e7a; font-style: italic; font-size: 11px; }
      .method-badge { font-size: 10px; padding: 1px 6px; border-radius: 3px; margin-left: 5px; }
      .method-root { background: rgba(62,207,142,.1); color: #3ecf8e; border: 1px solid rgba(62,207,142,.2); }
      .method-fallback { background: rgba(255,92,92,.1); color: #ff8080; border: 1px solid rgba(255,92,92,.2); }
    </style>
    <table class='vt'>
    <thead><tr>
      <th>Video ID</th><th>Segment</th><th>Time</th>
      <th>Original Caption</th><th>Masked Caption</th><th>Verb Label</th>
    </tr></thead><tbody>
    """)

    prev_vid = None
    for _, row in df_subset.iterrows():
        vid_cell = ''
        if row['video_id'] != prev_vid:
            vid_cell = f"<span class='vid-id'>{row['video_id']}</span>"
            prev_vid = row['video_id']

        masked = row['masked_caption']
        if '[MASK]' in masked:
            masked_html = masked.replace('[MASK]', "<span class='mask-tok'>[MASK]</span>")
        else:
            masked_html = f"<span class='muted'>{masked}</span>"

        if row['extracted_verb']:
            m_cls = 'method-root' if row['extraction_method'] == 'root' else 'method-fallback'
            verb_html = (f"<span class='verb-badge'>{row['extracted_verb']}</span>"
                         f"<span class='method-badge {m_cls}'>{row['extraction_method']}</span>")
        else:
            verb_html = "<span class='muted'>—</span>"

        rows.append(f"""<tr>
          <td>{vid_cell}</td>
          <td><span class='muted'>{row.get('segment_id', '')}</span></td>
          <td style='white-space:nowrap;color:#555e7a'>
            {float(row['start_sec']):.1f}s → {float(row['end_sec']):.1f}s</td>
          <td>{row['caption']}</td>
          <td>{masked_html}</td>
          <td>{verb_html}</td>
        </tr>""")

    rows.append('</tbody></table>')
    return ''.join(rows)

display(HTML(build_html_table(preview_df)))


Video ID,Segment,Time,Original Caption,Masked Caption,Verb Label
GLd3aX16zBg,GLd3aX16zBg_000,90.0s → 102.0s,spread margarine on two slices of white bread,[MASK] margarine on two slices of white bread,spreadroot
,GLd3aX16zBg_001,114.0s → 127.0s,place a slice of cheese on the bread,[MASK] a slice of cheese on the bread,placeroot
,GLd3aX16zBg_002,132.0s → 138.0s,place the bread slices on top of each other and place in a hot pan,[MASK] the bread slices on top of each other and place in a hot pan,placeroot
,GLd3aX16zBg_003,139.0s → 145.0s,flip the sandwich over and press down,[MASK] the sandwich over and press down,fliproot
,GLd3aX16zBg_004,173.0s → 174.0s,cut the sandwich in half diagonally,[MASK] the sandwich in half diagonally,cutroot
xHr8X2Wpmno,xHr8X2Wpmno_000,47.0s → 60.0s,pick the ends off the verdalago,[MASK] the ends off the verdalago,pickroot
,xHr8X2Wpmno_001,67.0s → 89.0s,combine lemon juice sumac garlic salt and oil in a bowl,[MASK] lemon juice sumac garlic salt and oil in a bowl,combineroot
,xHr8X2Wpmno_002,91.0s → 98.0s,chop lettuce and place it in a bowl,[MASK] lettuce and place it in a bowl,choproot
,xHr8X2Wpmno_003,99.0s → 137.0s,add verdalago pepper cucumbers tomatoes herbs and onions to the lettuce in the bowl,[MASK] verdalago pepper cucumbers tomatoes herbs and onions to the lettuce in the bowl,addroot
,xHr8X2Wpmno_004,153.0s → 162.0s,pour the dressing over the salad and mix,[MASK] the dressing over the salad and mix,pourroot


In [10]:
# Captions where no verb was found — inspect for patterns
no_verb = df[df['extracted_verb'].isna()][['video_id', 'caption']]
print(f'Captions with no verb extracted: {len(no_verb):,}')
if len(no_verb):
    print('\nSample (first 10):')
    for _, r in no_verb.head(10).iterrows():
        print(f'  [{r["video_id"]}] {r["caption"]}')


Captions with no verb extracted: 687

Sample (first 10):
  [V53XmPeyjIU] place chicken on a plate or tray and season generously with mixed spices
  [V53XmPeyjIU] season the flour with salt pepper and celery salt
  [V53XmPeyjIU] coat the chicken pieces in the flour
  [V53XmPeyjIU] place chicken in hot oil and fry until golden brown
  [V53XmPeyjIU] fry dark meats together and white meats together
  [H_9oM1Y60lU] grill the  patties
  [_xOx9hkJoBk] mix of ingredients in the bowl
  [_xOx9hkJoBk] coat the squid rings with the mixture in the bowl
  [gEYyWqs1oL0] place wasabi next to the tuna
  [Ap3g0IRcjzU] season with a little salt and pepper


## 9. Save outputs (NOT YET, SOME STILL HAVENT BEEN LABELED)

In [11]:
# Full annotated CSV — only rows where spaCy identified a verb, ordered as requested
out_df = (
    df[df['extracted_verb'].notna()]
    [[
        'caption',          # original caption
        'video_id',         # video the segment belongs to
        'segment_id',       # segment within that video
        'extracted_verb',   # verb identified by spaCy
        'masked_caption',   # caption with [MASK] applied
    ]]
    .sort_values(['video_id', 'segment_id'])
    .reset_index(drop=True)
)

out_path = os.path.join(PATHS['annotations'], 'segments_verbs.csv')
out_df.to_csv(out_path, index=False)
print(f'Saved → {out_path}  ({len(out_df):,} rows)')

# Stratified spot-check sample for manual validation
spotcheck = (
    df[df['extracted_verb'].notna()]
    .groupby(['subset', 'recipe_type'], group_keys=False)
    .apply(lambda x: x.sample(
        min(len(x), max(1, int(len(x) / len(df) * 100))),
        random_state=42
    ))
    .sample(min(100, len(df)), random_state=42)
    [[
        'caption',
        'video_id',
        'segment_id',
        'extracted_verb',
        'masked_caption',
        'extraction_method',
    ]]
    .sort_values(['video_id', 'segment_id'])
    .reset_index(drop=True)
)
spotcheck['manual_correct'] = ''  # fill in: 1 = correct, 0 = wrong
spotcheck['manual_notes']   = ''  # free text notes

spot_path = os.path.join(PATHS['annotations'], 'spotcheck_sample.csv')
spotcheck.to_csv(spot_path, index=False)
print(f'Saved → {spot_path}  ({len(spotcheck)}-row stratified validation sample)')


Saved → /home/ubuntu/youcook2-videos/annotations/segments_verbs.csv  (13,029 rows)
Saved → /home/ubuntu/youcook2-videos/annotations/spotcheck_sample.csv  (100-row stratified validation sample)


## Done ✓

Proceed to **Notebook 4** for frame extraction and embedding.

**What was produced:**
- `annotations/segments_verbs.csv` — full table with `extracted_verb`, `extraction_method`, `masked_caption`
- `annotations/spotcheck_sample.csv` — 100 rows for manual precision validation
- `annotations/verb_extraction_report.txt` — methods-section text
- `annotations/verb_frequency.png` — top-30 verb bar chart

**Before submitting your paper:** fill in `manual_correct` in `spotcheck_sample.csv`
and report `Precision = X/100` in your methods section. That single number
is what reviewers need to trust your ground-truth labels.
